# Velutina — Detector con cajas (YOLO, dataset Kaggle Hornet3000+)

Guia en espanol llano. No hay que programar: solo ejecutar las celdas en orden.

Este cuaderno entrena un **detector** (no un clasificador): dibuja **cajas**
alrededor de cada avispa y dice de que tipo es: `velutina`, `crabro` o `vulgaris`.

- Dataset: **Hornet3000+ (hornet3000)** de Kaggle, con cajas YOLO ya hechas
  (clase 0 = velutina, 1 = crabro, 2 = vulgaris).
- Referencia: https://www.kaggle.com/datasets/marcoryvandijk/vespa-velutina-v-crabro-vespulina-vulgaris
  y guia de entreno https://github.com/vespCV/hornet3000 (yolov10 en Colab).
- Todo pasa en Colab gratis. En tu ordenador solo quedaran 2 ficheros (~6 MB).
- Cada celda se puede volver a ejecutar sin romper nada (idempotente) e imprime su progreso.


## Celda 1 — GPU y ajustes (configuracion arriba del todo)

Comprueba si hay GPU gratis y deja los ajustes en 4 variables claras.


In [ ]:
# --- Celda 1: GPU + instalacion + configuracion ---
import torch
print('PyTorch:', torch.__version__)
print('Hay GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Nombre GPU:', torch.cuda.get_device_name(0))
else:
    print('Sin GPU: se usara CPU con pocos epochs (modo prueba).')

# Instalamos la libreria de YOLO (solo una vez; si ya esta, no hace nada malo)
!pip install -q ultralytics
print('ultralytics instalado.')

# ===== CONFIGURACION (toca aqui si quieres) =====
MODEL = "yolov8n.pt"   # modelo detector de partida (yolov8n.pt, ~6 MB)
import torch as _t
EPOCHS = 30 if _t.cuda.is_available() else 5   # 30 con GPU, 5 de prueba si CPU
IMGSZ = 640   # tamano de imagen de entrada
BATCH = 16    # imagenes por lote
print(f'Config: MODEL={MODEL} EPOCHS={EPOCHS} IMGSZ={IMGSZ} BATCH={BATCH}')


## Celda 2 — Clave de Kaggle (la subes tu a mano)

El dataset Hornet3000+ esta en Kaggle y pide entrar con tu cuenta.
El cuaderno NO trae ninguna clave: la subes tu como fichero.


In [ ]:
# --- Celda 2: credencial kaggle.json (subida manual por el usuario) ---
from pathlib import Path
import os, stat, subprocess

KAGGLE_JSON = Path("/content/kaggle.json")  # asi se llama el fichero que subes a Colab
print('Buscando /content/kaggle.json ...')
if not KAGGLE_JSON.exists():
    print('')
    print('FALTA tu fichero kaggle.json. Haz esto (2 minutos, una sola vez):')
    print('  1. Entra en https://www.kaggle.com y entra con tu cuenta (o crea una gratis).')
    print('  2. Clica tu avatar (arriba derecha) -> Settings.')
    print('  3. Baja a la seccion API y clica Create New Token.')
    print('  4. Se descarga un fichero llamado kaggle.json a tu ordenador.')
    print('  5. En Colab, abre el panel Archivos (carpeta a la izquierda).')
    print('  6. Arrastra kaggle.json al panel (queda en /content/kaggle.json).')
    print('  7. Vuelve a ejecutar esta celda.')
    print('')
    raise SystemExit('Falta /content/kaggle.json: subelo al panel Archivos de Colab y re-ejecuta esta celda.')

# Si existe: lo colocamos donde Kaggle lo espera, con permisos solo-lectura para ti
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
dest = Path(os.path.expanduser('~/.kaggle/kaggle.json'))
dest.write_bytes(KAGGLE_JSON.read_bytes())
os.chmod(dest, 0o600)
print(f'kaggle.json copiado a {dest} con permisos 600.')
!pip install -q kaggle
print('Cliente kaggle listo.')


## Celda 3 — Descarga del dataset Hornet3000+ y data.yaml

Baja el zip de Kaggle y lo descomprime en `data_k`.
Luego mira como viene por dentro y prepara el `data.yaml` que pide YOLO.


In [ ]:
# --- Celda 3: descarga Hornet3000+ y verificacion de estructura ---
from pathlib import Path
import subprocess, os

# Hornet3000+ (hornet3000): cajas YOLO, clases 0=velutina 1=crabro 2=vulgaris
DATA_DIR = Path('data_k')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Descargando dataset de Kaggle (solo la primera vez pesa; luego se reutiliza)...')
r = subprocess.run(['kaggle', 'datasets', 'download',
                    '-d', 'marcoryvandijk/vespa-velutina-v-crabro-vespulina-vulgaris',
                    '-p', str(DATA_DIR), '--unzip'], capture_output=True, text=True)
print(r.stdout[-2000:] if r.stdout else '')
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit('Fallo la descarga de Kaggle. Revisa tu kaggle.json y re-ejecuta.')
print('Descarga OK.')

# Arbol defensivo: ensenamos que trae el zip (cada zip puede traer otro orden de carpetas)
print('--- Arbol de data_k (primeros 60) ---')
all_paths = sorted([p for p in DATA_DIR.rglob('*') if p.is_file()])
for p in all_paths[:60]:
    print(' ', p.relative_to(DATA_DIR))
print(f'Total ficheros: {len(all_paths)}')

imgs = [p for p in all_paths if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
lbls = [p for p in all_paths if p.suffix.lower() == '.txt' and p.name != 'data.yaml']
print(f'Imagenes encontradas (*.jpg/*.png): {len(imgs)}')
print(f'Etiquetas YOLO encontradas (*.txt): {len(lbls)}')
if not imgs:
    raise SystemExit('No se encontraron imagenes en data_k. Mira el arbol de arriba y adapta las rutas.')

# Localizar o crear data.yaml con names [vespa_velutina, vespa_crabro, vespula_vulgaris]
cands = list(DATA_DIR.rglob('data.yaml')) + list(DATA_DIR.rglob('*.yaml'))
YAML = None
for c in cands:
    t = c.read_text(encoding='utf-8', errors='ignore')
    if 'train' in t and 'names' in t:
        YAML = c
        print(f'data.yaml del propio zip: {c}')
        print(t[:800])
        break
if YAML is None:
    print('El zip no trae data.yaml valido: creo uno apuntando a lo encontrado.')
    # Heuristica: carpetas images/train|val|test y labels/train|val|test si existen; si no, raiz
    def pick(kind, split):
        p = DATA_DIR / kind / split
        if p.is_dir(): return str(p.resolve())
        p2 = next((d for d in DATA_DIR.rglob(split) if d.is_dir() and kind in [x.name for x in [d.parent] + list(d.parents)]), None)
        return str(p2.resolve()) if p2 else ''
    tr_i, va_i = pick('images', 'train'), pick('images', 'val')
    tr_l, va_l = pick('labels', 'train'), pick('labels', 'val')
    if not tr_i:  # layout plano: todo en raiz
        tr_i = va_i = str(DATA_DIR.resolve())
    YAML = DATA_DIR / 'data.yaml'
    YAML.write_text(
        f"path: {DATA_DIR.resolve()}\n"
        f"train: {tr_i or str(DATA_DIR.resolve())}\n"
        f"val: {va_i or tr_i or str(DATA_DIR.resolve())}\n"
        'names: [vespa_velutina, vespa_crabro, vespula_vulgaris]\n', encoding='utf-8')
    print(f'data.yaml creado en {YAML}:')
    print(YAML.read_text(encoding='utf-8'))
print(f'YAML final: {YAML}')


## Celda 4 — Entrenamiento del detector

Entrena `yolov8n.pt` con tus cajas. Tarda ~1 h con GPU gratis.


In [ ]:
# --- Celda 4: entrena YOLO('yolov8n.pt') detector ---
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Entrenando {MODEL} con device={DEVICE}, epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}')
print(f'data={str(YAML)}')
model = YOLO(MODEL)  # detector yolov8n.pt (no el cls: este localiza + clasifica)
model.train(data=str(YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE)
print('Entrenamiento terminado. Pesos en runs/detect/train/weights/best.pt')


## Celda 5 — Evaluacion oficial (mAP, precision, recall)


In [ ]:
# --- Celda 5: model.val() -> mAP50, mAP50-95, precision/recall + por clase ---
from ultralytics import YOLO
from pathlib import Path

BEST = Path('runs/detect/train/weights/best.pt')
print(f'Evaluando: {BEST} (existe: {BEST.exists()})')
model = YOLO(str(BEST) if BEST.exists() else MODEL)
metrics = model.val(data=str(YAML))
print('--- Metricas globales ---')
print(f"mAP50:    {float(metrics.box.map50):.4f}")
print(f"mAP50-95: {float(metrics.box.map):.4f}")
print(f"precision:{float(metrics.box.mp):.4f}  recall:{float(metrics.box.mr):.4f}")
print('--- Por clase (0=vespa_velutina, 1=vespa_crabro, 2=vespula_vulgaris) ---')
try:
    import numpy as np
    names = metrics.names
    ap50 = np.array(metrics.box.ap50)
    ap = np.array(metrics.box.ap)
    p = np.array(metrics.box.p)
    r = np.array(metrics.box.r)
    for i, n in names.items():
        print(f"  clase {i} ({n}): AP50={ap50[i]:.4f} AP50-95={ap[i]:.4f} P={p[i]:.4f} R={r[i]:.4f}")
except Exception as e:
    print(f'(detalle por clase no disponible: {e})')


## Celda 6 — Prueba propia: ¿cuantas velutinas pilla y cuantos FP duros mete?

La metrica oficial esta bien, pero aqui medimos lo que te importa:
de cada 100 velutinas, ¿cuantas detecta? Y en fotos SIN velutina,
¿cuantas cajas `velutina` mete por error (crabro / vulgaris / fondo)?


In [ ]:
# --- Celda 6: recall@0.5 de velutina + FP duros; guarda resultados_det.txt ---
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

CONF = 0.5  # umbral de confianza para contar una caja como deteccion
VEL = 0     # clase 0 = vespa_velutina
BEST = Path('runs/detect/train/weights/best.pt')
model = YOLO(str(BEST) if BEST.exists() else MODEL)

val_txt = YAML.parent / 'val.txt'
if val_txt.exists():
    val_imgs = [Path(l.strip()) for l in val_txt.read_text().splitlines() if l.strip()]
else:  # reconstruir val desde el yaml: carpeta(s) de validacion
    import yaml as _y
    cfg = _y.safe_load(YAML.read_text())
    v = cfg.get('val')
    vpaths = [v] if isinstance(v, str) else list(v or [])
    val_imgs = []
    for vp in vpaths:
        vp = Path(vp)
        if vp.is_dir():
            val_imgs += sorted([p for p in vp.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
        elif vp.is_file() and vp.suffix == '.txt':
            val_imgs += [Path(l.strip()) for l in vp.read_text().splitlines() if l.strip()]
print(f'Imagenes de val a revisar: {len(val_imgs)}')

def label_has_velutina(img_path):
    # espejo images->labels buscando el .txt YOLO hermano
    cand = Path(str(img_path).replace('images', 'labels')).with_suffix('.txt')
    if not cand.exists():
        cand = img_path.with_suffix('.txt')
    if not cand.exists(): return None  # sin etiqueta: no cuenta
    for ln in cand.read_text().splitlines():
        if ln.strip().split():
            try:
                if int(float(ln.strip().split()[0])) == VEL: return True
            except ValueError: pass
    return False

n_vel_imgs = n_vel_ok = n_fp_duros = n_sin_etiqueta = 0
fp_origen = Counter()
for img in val_imgs:
    truth = label_has_velutina(img)
    if truth is None:
        n_sin_etiqueta += 1
        continue
    res = model.predict(str(img), conf=CONF, verbose=False)[0]
    pred_vel = any(int(c) == VEL for c in res.boxes.cls.tolist()) if len(res.boxes) else False
    if truth:
        n_vel_imgs += 1
        if pred_vel: n_vel_ok += 1
    else:
        if pred_vel:
            n_fp_duros += 1
            # origen: que clase real trae la imagen (crabro/vulgaris/fondo)
            cand = Path(str(img).replace('images', 'labels')).with_suffix('.txt')
            t = cand.read_text() if cand.exists() else ''
            cls = sorted({l.split()[0] for l in t.splitlines() if l.strip().split()}) if t.strip() else []
            fp_origen[','.join(cls) if cls else 'fondo/sin_caja'] += 1

recall_vel = (n_vel_ok / n_vel_imgs) if n_vel_imgs else 0.0
print(f'--- Resultado propio (conf={CONF}) ---')
print(f'Imagenes val con velutina: {n_vel_imgs}; detectadas: {n_vel_ok} -> recall velutina = {recall_vel:.3f}')
print(f'Cajas velutina en imagenes SIN velutina (FP duros): {n_fp_duros}')
print(f'Origen FP duros: {dict(fp_origen)}')
print(f'(sin etiqueta YOLO, ignoradas: {n_sin_etiqueta})')

OUT = Path('resultados_det.txt')
OUT.write_text(
    f"mAP50={float(metrics.box.map50):.4f} mAP50-95={float(metrics.box.map):.4f} "
    f"P={float(metrics.box.mp):.4f} R={float(metrics.box.mr):.4f}\n"
    f"recall_velutina@{CONF}={recall_vel:.4f} ({n_vel_ok}/{n_vel_imgs})\n"
    f"FP_duros_velutina_en_no_velutina={n_fp_duros} origen={dict(fp_origen)}\n"
    f"conf={CONF} val_imgs={len(val_imgs)} modelo={MODEL} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH}\n",
    encoding='utf-8')
print(f'Guardado {OUT} ({OUT.stat().st_size} bytes):')
print(OUT.read_text(encoding='utf-8'))


## Celda 7 — Exportar a ONNX y descargar (2 ficheros, ~6 MB)

Deja el modelo listo para la Jetson y te dice donde poner cada fichero.


In [ ]:
# --- Celda 7: exporta ONNX + tamanos + instrucciones de descarga ---
from ultralytics import YOLO
from pathlib import Path

BEST = Path('runs/detect/train/weights/best.pt')
model = YOLO(str(BEST))
onnx = model.export(format='onnx')  # crea best.onnx junto al best.pt
print(f'ONNX exportado: {onnx}')
for f in [BEST, Path(str(onnx)), Path('resultados_det.txt')]:
    print(f"  {f} -> {f.stat().st_size/1e6:.1f} MB" if f.exists() else f"  {f} NO existe")
print('')
print('DESCARGA (panel Archivos de Colab, click derecho -> Descargar):')
print('  1. runs/detect/train/weights/best.pt  -> guardalo en tu repo como jetson/ai/yolov8n_det.pt')
print('     (nombre DISTINTO a los cls: NO pises yolov8n_velutina.pt ni yolov8n_velutina_v2.pt)')
print('  2. resultados_det.txt                -> guardalo junto al .pt para comparar')
print('En total ~6 MB. El dataset y los pesos intermedios se quedan en Colab.')


## Detector vs clasificador (lectura honesta, 1 minuto)

El **clasificador** (v1/v2) responde a una foto recortada: *¿que bicho es este?*.
El **detector** (este cuaderno) responde a una escena entera: *¿donde hay bichos
y de que tipo es cada uno?* (cajas + clase). Para el apiario el detector es lo
que vale: localiza en la escena.

Pero ojo: este detector se entrena con **fotos de observadores** (bicho grande,
centrado, quieto), no con vuelo real en tu apiario (bicho pequeno, movido,
con tu fondo y tu luz). Asi que sus numeros en Kaggle NO son numeros en campo.

Que falta para el disparo (no disparar todavia solo con esto):

1. **Validacion en escena real**: graba tu piquera/fondo, anota unas decenas
   de fotos y mide recall/FP ahi (el `resultados_det.txt` de Kaggle no vale).
2. **Calibracion del umbral**: sube `conf` hasta que los FP duros
   (crabro/vulgaris/fondo) sean ~0 y mira cuantas velutinas sigues pillando.
3. **Regla de disparo**: exige varias detecciones seguidas (track) antes de
   actuar; una sola caja no basta.

Referencias: dataset https://www.kaggle.com/datasets/marcoryvandijk/vespa-velutina-v-crabro-vespulina-vulgaris
 y guia https://github.com/vespCV/hornet3000.
